**Predição de Aprovação de Pedidos de Cartão de Crédito Utilizando
Aprendizado de Máquina**  

Este projeto tem como objetivo desenvolver um modelo de aprendizado
de máquina capaz de prever se um pedido de cartão de crédito deve ser
aprovado com base em informações pessoais e financeiras dos
solicitantes.

Utilizando técnicas de classificação supervisionadas e uma base
de dados composta por atributos como renda, ocupação, estado civil e histórico
de trabalho, o modelo busca classificar os clientes como "bons" ou "maus"
pagadores, auxiliando instituições financeiras na tomada de decisão quanto à
concessão de crédito.

O projeto aborda as etapas de pré-processamento de
dados, análise exploratória, seleção de atributos, modelagem e avaliação de
desempenho, utilizando diferentes algoritmos.


#**0 — Importanto base de dados**

In [ ]:
import pandas as pd

In [ ]:
df_credit = pd.read_csv("../data/raw/credit_record.csv")
df_credit.head()

,ID,MONTHS_BALANCE,STATUS
0,5001711,0,X
1,5001711,-1,0
2,5001711,-2,0
3,5001711,-3,0
4,5001712,0,C


In [ ]:
df_application = pd.read_csv("../data/raw/application_record.csv")
df_application.head()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0


# **1 — Análise Exploratória de Dados - credit_record**

| Coluna           | Significado    | Interpretação                            |
| ---------------- | -------------- | ---------------------------------------- |
| `ID`             | Identification | Identificador do cliente                 |
| `MONTHS_BALANCE` | Months Balance | Mês de referência do registro de crédito |
| `STATUS`         | Credit Status  | Situação do crédito naquele mês          |


In [ ]:
df_credit.shape

(1048575, 3)

In [ ]:
#quantidade de registros e informação de nulos
df_credit.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 3 columns):
 #   Column          Non-Null Count    Dtype 
---  ------          --------------    ----- 
 0   ID              1048575 non-null  int64 
 1   MONTHS_BALANCE  1048575 non-null  int64 
 2   STATUS          1048575 non-null  object
dtypes: int64(2), object(1)
memory usage: 24.0+ MB


In [ ]:
df_credit['STATUS'].unique()

array(['X', '0', 'C', '1', '2', '3', '4', '5'], dtype=object)

In [ ]:
#registros duplicados
df_credit.duplicated().sum()

np.int64(0)

In [ ]:
df_credit.duplicated(
    subset=['ID', 'MONTHS_BALANCE']
).sum()

np.int64(0)

In [ ]:
#quantos meses existem de registro
df_credit['MONTHS_BALANCE'].unique()

array([  0,  -1,  -2,  -3,  -4,  -5,  -6,  -7,  -8,  -9, -10, -11, -12,
       -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25,
       -26, -27, -28, -29, -30, -31, -32, -33, -34, -35, -36, -37, -38,
       -39, -40, -41, -42, -43, -44, -45, -46, -47, -48, -49, -50, -51,
       -52, -53, -54, -55, -56, -57, -58, -59, -60])

In [ ]:
#VERIFICANDO SE TODOS OS CLIENTES TÊM 61 MESES DE REGISTRO
meses_por_cliente = df_credit.groupby('ID')['MONTHS_BALANCE'].nunique()
meses_por_cliente.value_counts().sort_index()


,count
MONTHS_BALANCE,
1,399
2,1088
3,1163
4,1339
5,1227
...,...
57,281
58,238
59,242




```
# Observações:
1- não há valores duplicados de registros
2- nem todo cliente ID tem 61 MONTHS_BALANCE de informação. Impossibilitando target com período de cura
```



# **2 — Definição da variável alvo TARGET - credit_record**

0 = menor risco

1 = maior risco


| STATUS | Significado           | Interpretação                                        |
| ------ | --------------------- | ---------------------------------------------------- |
| `C`    | Paid/closed           | Crédito quitado/sem saldo naquele mês                |
| `0`    | 1–29 days past due    | Atraso de 1 a 29 dias                                |
| `1`    | 30–59 days past due   | Atraso de 30 a 59 dias                               |
| `2`    | 60–89 days past due   | Atraso de 60 a 89 dias                               |
| `3`    | 90–119 days past due  | Atraso de 90 a 119 dias                              |
| `4`    | 120–149 days past due | Atraso de 120 a 149 dias                             |
| `5`    | 150+ days past due    | Atraso de 150 dias ou mais                           |
| `X`    | No loan for the month | Sem informação de crédito/sem empréstimo naquele mês |


In [ ]:
#quantidade por tipo de status
df_credit['STATUS'].value_counts(dropna=False)

,count
STATUS,
C,442031
0,383120
X,209230
1,11090
5,1693
2,868
3,320
4,223


In [ ]:
#percentual por tipo de status
df_credit['STATUS'].value_counts(normalize=True, dropna=False) * 100

,proportion
STATUS,
C,42.155401
0,36.537205
X,19.953747
1,1.057626
5,0.161457
2,0.082779
3,0.030518
4,0.021267



```
# Observações:
1- forte desbalanceamento da base, grande parte dos clientes está com créditos quitados ou sem informação
```



###   **2.1 Testando a Regra de negócio 1 - RECORRÊNCIA**: considerar maior risco quem teve mais de 50% de ocorrências com 60 dias ou mais de atraso
STATUS >= 2 **nos últimos =~5 anos (61 meses da base)**

In [ ]:
#transformando todos os registros de STATUS em uma variável numérica
df_credit['STATUS_NUM'] = df_credit['STATUS'].replace({
    'C': -1,
    'X': -1
}).astype(int)

In [ ]:
# criando target para status > = 2
target = (
    df_credit
    .groupby('ID')
    .agg(
        TOTAL_MESES=('STATUS_NUM', 'count'),
        MESES_STATUS_2_MAIS=(
            'STATUS_NUM',
            lambda x: (x >= 2).sum()
        )
    )
    .reset_index()
)
target.head()

,ID,TOTAL_MESES,MESES_STATUS_2_MAIS
0,5001711,4,0
1,5001712,19,0
2,5001713,22,0
3,5001714,15,0
4,5001715,60,0


In [ ]:
#calculando percentual em que o cliente ficou com status >=2
target['PERC_STATUS_2_MAIS'] = (
    target['MESES_STATUS_2_MAIS'] /
    target['TOTAL_MESES']
)
target.head()

,ID,TOTAL_MESES,MESES_STATUS_2_MAIS,PERC_STATUS_2_MAIS
0,5001711,4,0,0.0
1,5001712,19,0,0.0
2,5001713,22,0,0.0
3,5001714,15,0,0.0
4,5001715,60,0,0.0


In [ ]:
#criando target para cliente com mais de 50% de registros com status>=2

#0 = menor risco 1 = maior risco



target['TARGET'] = (
    target['PERC_STATUS_2_MAIS'] > 0.50
).astype(int)
target.head()

,ID,TOTAL_MESES,MESES_STATUS_2_MAIS,PERC_STATUS_2_MAIS,TARGET
0,5001711,4,0,0.0,0
1,5001712,19,0,0.0,0
2,5001713,22,0,0.0,0
3,5001714,15,0,0.0,0
4,5001715,60,0,0.0,0


In [ ]:
# identificando TARGET final por quantidade
target['TARGET'].value_counts()

,count
TARGET,
0,45934
1,51


In [ ]:
# identificando TARGET final por percentual
target['TARGET'].value_counts(normalize=True) * 100

,proportion
TARGET,
0,99.889094
1,0.110906


```
# Observações:
1- Regra de negócio 1 - RECORRÊNCIA : desbalanceda, considerando quase todos os clientes como de menor risco 99,8%
```

###   **2.2 Testando a Regra de negócio 2** - considerar de maior risco o cliente que apresentou STATUS >= 2 (atraso de 60 dias ou mais) em pelo menos 1 mês nos últimos =~5 anos (61 meses da base)


**TARGET = 1 cliente que apresentou pelo menos um atraso grave (STATUS >= 2) durante o período de =~ 5 anos.**


In [ ]:
# criando variável com status máximo de cada cliente (ID)
target = (
    df_credit
    .groupby('ID')['STATUS_NUM']
    .max()
    .reset_index()
)

In [ ]:
#criando coluna Target para clientes com status maior ou igual a 60 dias de atraso
target['TARGET'] = (
    target['STATUS_NUM'] >= 2
).astype(int)

In [ ]:
#verificando percentual por target
target['TARGET'].value_counts(normalize=True) * 100

,proportion
TARGET,
0,98.549527
1,1.450473


In [ ]:
clientes_status = (
    df_credit
    .groupby('ID')['STATUS_NUM']
    .max()
)

clientes_status.value_counts().sort_index()

,count
STATUS_NUM,
-1,5953
0,34682
1,4683
2,336
3,88
4,48
5,195


In [ ]:
(clientes_status >= 2).value_counts(normalize=True) * 100

,proportion
STATUS_NUM,
False,98.549527
True,1.450473


```
# Observações:
1- Regra de negócio 2 : desbalanceda, considerando quase todos os clientes como de menor risco 98,5%

*   Item da lista
*   Item da lista


```

In [ ]:
#percentual por tipo de status
df_credit['STATUS'].value_counts(normalize=True, dropna=False) * 100

,proportion
STATUS,
C,42.155401
0,36.537205
X,19.953747
1,1.057626
5,0.161457
2,0.082779
3,0.030518
4,0.021267


```
# Observações:
1- incluir mais clientes do status 0 para balancear a base
```

###   **2.3 Testando a Regra de negócio 3**

- **considerar a reincidência de atraso**. Será de maior risco o cliente que apresentou STATUS = 1 (atraso de 30 dias ou mais) pelo menos 2 vezes durantes =~ 5 anos - 60 meses


- **considerar ocorrência de atraso grave**. Também será de maior risco o cliente que apresentou STATUS >= 2 (atraso de 60 dias ou mais)  pelo vezes 1 vez durantes =~ 5 anos - 60 meses


- **considerar a reincidência de atraso**. Será de maior risco o cliente que apresentou STATUS = 0 (atraso de 1 a 29)  pelo menos 6 vezes durante =~ 5 anos - 60 meses


TARGET = 1 se:

STATUS = 1 em ≥ 2 meses OU

STATUS ≥ 2 em ≥ 1 mês OU

STATUS = 0 em ≥ 6 meses**

In [ ]:
TARGET3 = (
    df_credit
    .groupby('ID')
    .agg(
        TOTAL_MESES=('STATUS_NUM', 'count'),

        MESES_STATUS_0=(
            'STATUS_NUM',
            lambda x: (x == 0).sum()
        ),

        MESES_STATUS_1=(
            'STATUS_NUM',
            lambda x: (x == 1).sum()
        ),

        MESES_STATUS_2_MAIS=(
            'STATUS_NUM',
            lambda x: (x >= 2).sum()
        )
    )
    .reset_index()
)

In [ ]:
#criando a variável target para a regra de negócio 3
TARGET3['TARGET'] = (
    (TARGET3['MESES_STATUS_1'] >= 2) |
    (TARGET3['MESES_STATUS_2_MAIS'] >= 1) |
    (TARGET3['MESES_STATUS_0'] >= 6)
).astype(int)

In [ ]:
TARGET3['TARGET'].value_counts(normalize=True) * 100

,proportion
TARGET,
1,57.747091
0,42.252909


In [ ]:
#verificando registros duplicados
TARGET3.duplicated(
    subset=['ID', 'TARGET']
).sum()

np.int64(0)

```
# Observações:
1- Regra de negócio 3 a menos desbalanceada.

In [ ]:
TARGET3.head()

,ID,TOTAL_MESES,MESES_STATUS_0,MESES_STATUS_1,MESES_STATUS_2_MAIS,TARGET
0,5001711,4,3,0,0,0
1,5001712,19,10,0,0,1
2,5001713,22,0,0,0,0
3,5001714,15,0,0,0,0
4,5001715,60,0,0,0,0
